In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats
from scipy.stats import mstats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import streamlit as st

In [2]:
_candidatos = [
    os.path.join(os.getcwd(), "dataset_saude_brasil.csv"),
    r"C:\Projetos\pos ia\dataset_saude_brasil.csv",
]
CAMINHO_DADOS = next((p for p in _candidatos if os.path.isfile(p)), None)
if CAMINHO_DADOS is None:
    raise FileNotFoundError("dataset_saude_brasil.csv não encontrado.")
df = pd.read_csv(CAMINHO_DADOS)
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\rafaela.santana\AppData\Local\Temp\ipykernel_25904\1426072249.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv('C:\Projetos\pos ia\dataset_saude_brasil.csv')


# Análise Exploratória de Dados (EDA)
Vamos entender melhor a distribuição dos dados e a correlação entre as principais variáveis clínicas antes de aplicar o modelo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de estilo
sns.set_theme(style="whitegrid")

# 1. Distribuição de algumas variáveis chaves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['Idade'], bins=20, kde=True, color='skyblue', ax=axes[0])
axes[0].set_title('Distribuição de Idade')

sns.histplot(df['IMC'], bins=20, kde=True, color='salmon', ax=axes[1])
axes[1].set_title('Distribuição de IMC')

sns.countplot(data=df, x='Risco_Doenca', palette='viridis', order=['Baixo', 'Moderado', 'Alto', 'Muito Alto'], ax=axes[2])
axes[2].set_title('Distribuição de Risco de Doença')

plt.tight_layout()
plt.show()

# 2. Mapa de Correlações Globais (Variáveis Numéricas)
plt.figure(figsize=(12, 8))
cols_numericas = ['Idade', 'IMC', 'Passos_Diarios', 'Horas_Sono', 'Agua_Litros', 'Calorias', 'Pressao_Sistolica', 'Pressao_Diastolica', 'Colesterol', 'Frequencia_Cardiaca_Repouso']

# Selecionar apenas numéricas presentes
cols_presentes = [c for c in cols_numericas if c in df.columns]

if cols_presentes:
    corr = df[cols_presentes].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Mapa de Correlação das Variáveis Chaves')
    plt.show()

In [ ]:
# --- Tratamento de outliers (winsorização + IQR + cap por z-score) e engenharia de features ---

NUM_COLS = [
    "Idade", "IMC", "Passos_Diarios", "Horas_Sono", "Agua_Litros", "Calorias",
    "Horas_Trabalho", "Frequencia_Cardiaca_Repouso", "Pressao_Sistolica",
    "Pressao_Diastolica", "Colesterol",
]


def clip_iqr(s: pd.Series, factor: float = 1.5) -> pd.Series:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    return s.clip(lo, hi)


def winsorize_series(s: pd.Series, limits=(0.01, 0.01)) -> pd.Series:
    arr = s.astype(float).values
    mask = np.isfinite(arr)
    if mask.sum() == 0:
        return s
    out = arr.copy()
    out[mask] = mstats.winsorize(arr[mask], limits=limits)
    return pd.Series(out, index=s.index)


def cap_zscore(s: pd.Series, z: float = 3.0) -> pd.Series:
    mu, sd = s.mean(), s.std()
    if sd == 0 or np.isnan(sd):
        return s
    lo, hi = mu - z * sd, mu + z * sd
    return s.clip(lo, hi)


def preparar_base(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Caudas longas: winsorização (1–99%) + recorte IQR
    for col in ["Passos_Diarios", "Calorias", "Colesterol", "IMC"]:
        df[col] = winsorize_series(df[col], limits=(0.01, 0.01))
        df[col] = clip_iqr(df[col])

    # Pressão e FC: limitar extremos por z-score (≈ 3 desvios)
    for col in ["Pressao_Sistolica", "Pressao_Diastolica", "Frequencia_Cardiaca_Repouso"]:
        df[col] = cap_zscore(df[col], z=3.0)

    return df


def engenharia_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Hipertensao"] = (df["Pressao_Sistolica"] > 140).astype(int)
    df["Impacto_IMC_Idade"] = df["IMC"] * df["Idade"]
    df["Stress_Trabalho"] = ((df["Horas_Trabalho"] > 10) & (df["Horas_Sono"] < 6)).astype(int)

    df["Fumante_Num"] = df["Fumante"].map({"Sim": 1, "Não": 0})
    df["Alcool_Num"] = df["Alcool"].map({"Baixo": 0, "Moderado": 1, "Alto": 2})
    df["Sexo_Num"] = df["Sexo"].map({"Masculino": 1, "Feminino": 0})
    df["Hist_Familiar_Num"] = df["Historico_Familiar"].map({"Sim": 1, "Não": 0})

    df["MAP"] = (df["Pressao_Sistolica"] + 2 * df["Pressao_Diastolica"]) / 3
    df["Pressao_Pulso"] = df["Pressao_Sistolica"] - df["Pressao_Diastolica"]
    df["FC_Elevada"] = (df["Frequencia_Cardiaca_Repouso"] > 90).astype(int)
    df["Calorias_por_1000_passos"] = df["Calorias"] / (df["Passos_Diarios"] / 1000 + 1)
    df["SonoIdeal_dist"] = (df["Horas_Sono"] - 7.5).abs()
    df["Sono_restrito"] = (df["Horas_Sono"] < 6).astype(int)
    df["Sedentario"] = (df["Passos_Diarios"] < 5000).astype(int)
    df["Imc_categoria"] = (
        pd.cut(
            df["IMC"],
            bins=[-np.inf, 18.5, 25, 30, np.inf],
            labels=[0, 1, 2, 3],
        ).astype(float)
    )
    df["Fumante_Alcool"] = df["Fumante_Num"] * df["Alcool_Num"]
    df["Idade_risco"] = (df["Idade"] >= 55).astype(int)
    df["Colesterol_alto"] = (df["Colesterol"] > 240).astype(int)
    df["Horas_trabalho_int"] = df["Horas_Trabalho"] * df["Horas_Sono"]
    df["Agua_x_passos"] = df["Agua_Litros"] * np.log1p(df["Passos_Diarios"])
    df["Score_habito_ruim"] = df["Fumante_Num"] + df["Alcool_Num"] + df["Sedentario"]

    return df


FEATURES_ML = [
    "Idade", "IMC", "Passos_Diarios", "Horas_Sono", "Agua_Litros", "Calorias", "Horas_Trabalho",
    "Fumante_Num", "Alcool_Num", "Pressao_Sistolica", "Hipertensao", "Impacto_IMC_Idade",
    "Sexo_Num", "Hist_Familiar_Num", "Colesterol", "Stress_Trabalho",
    "Frequencia_Cardiaca_Repouso", "MAP", "Pressao_Pulso", "FC_Elevada", "Calorias_por_1000_passos",
    "SonoIdeal_dist", "Sono_restrito", "Sedentario", "Imc_categoria", "Fumante_Alcool",
    "Idade_risco", "Colesterol_alto", "Horas_trabalho_int", "Agua_x_passos", "Score_habito_ruim",
]

df_ml = engenharia_features(preparar_base(df))
X = df_ml[FEATURES_ML]
y = df_ml["Risco_Doenca"]
print(X.shape, y.value_counts().to_dict())

In [ ]:
# --- Treino e comparação de modelos (holdout 80/20 estratificado) ---

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelos = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_leaf=4,
        class_weight="balanced", random_state=42,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=400, max_depth=18, min_samples_leaf=3,
        class_weight="balanced", random_state=42,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.08, random_state=42,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300, max_depth=8, learning_rate=0.06, random_state=42,
    ),
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, solver="lbfgs", class_weight="balanced", random_state=42,
        )),
    ]),
    "KNN (k=15)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance")),
    ]),
}

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    resultados.append({"modelo": nome, "acuracia": acc, "f1_macro": f1})

df_resultados = pd.DataFrame(resultados).sort_values("f1_macro", ascending=False).reset_index(drop=True)
print(df_resultados.to_string(index=False))

melhor = df_resultados.iloc[0]["modelo"]
print(f"\nMelhor modelo (F1 macro): {melhor}")

fig, ax = plt.subplots(figsize=(8, 4))
df_plot = df_resultados.sort_values("f1_macro")
ax.barh(df_plot["modelo"], df_plot["f1_macro"], color="steelblue")
ax.set_xlabel("F1 macro (teste)")
ax.set_title("Comparação de modelos após outliers + novas features")
plt.tight_layout()
plt.show()

# Relatório do melhor classificador (objeto já treinado)
modelo_ref = modelos[melhor]
y_ref = modelo_ref.predict(X_test)
print(classification_report(y_test, y_ref, digits=4))

In [7]:
import pandas as pd
import joblib
import streamlit as st
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import os

# --- CONFIGURAÇÃO DA PÁGINA STREAMLIT ---
st.set_page_config(page_title="CardioPredict AI", layout="wide", page_icon="🏥")

# --- 1. FUNÇÃO DE TREINAMENTO (SÓ RODA SE O MODELO NÃO EXISTIR) ---
def treinar_modelo():
    # Carregamento (ajuste o caminho se necessário)
    caminho_csv = r'C:\Projetos\pos ia\dataset_saude_brasil.csv'
    if not os.path.exists(caminho_csv):
        st.error(f"Arquivo CSV não encontrado em: {caminho_csv}")
        return None

    df = pd.read_csv(caminho_csv)

    # Tratamento de Dados
    cols_numericas = ['Passos_Diarios', 'Calorias', 'Colesterol']
    for col in cols_numericas:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Idade'] = df['Idade'].fillna(df['Idade'].median())
    df['IMC'] = df['IMC'].fillna(df['IMC'].median())
    df['Passos_Diarios'] = df['Passos_Diarios'].fillna(df['Passos_Diarios'].mean())
    df['Colesterol'] = df['Colesterol'].fillna(df['Colesterol'].median())
    df['Calorias'] = df['Calorias'].fillna(df['Calorias'].median())

    # Feature Engineering
    df['Hipertensao'] = df['Pressao_Sistolica'].apply(lambda x: 1 if x > 140 else 0)
    df['Impacto_IMC_Idade'] = df['IMC'] * df['Idade']
    df['Stress_Trabalho'] = ((df['Horas_Trabalho'] > 10) & (df['Horas_Sono'] < 6)).astype(int)

    # Tratamento de Categorias
    df['Fumante_Num'] = df['Fumante'].map({'Sim': 1, 'Não': 0})
    df['Alcool_Num'] = df['Alcool'].map({'Baixo': 0, 'Moderado': 1, 'Alto': 2})
    df['Sexo_Num'] = df['Sexo'].map({'Masculino': 1, 'Feminino': 0})
    df['Hist_Familiar_Num'] = df['Historico_Familiar'].map({'Sim': 1, 'Não': 0})

    # Seleção de Features
    features = [
        'Idade', 'IMC', 'Passos_Diarios', 'Horas_Sono', 'Agua_Litros', 
        'Fumante_Num', 'Alcool_Num', 'Pressao_Sistolica', 'Hipertensao', 
        'Impacto_IMC_Idade', 'Sexo_Num', 'Hist_Familiar_Num', 'Colesterol',
        'Stress_Trabalho'
    ]

    X = df[features]
    y = df['Risco_Doenca']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Modelo
    modelo = RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_leaf=4, 
        class_weight='balanced', random_state=42
    )
    modelo.fit(X_train, y_train)
    
    # Salva o modelo
    joblib.dump(modelo, 'modelo_risco_saude.pkl')
    return modelo

# --- 2. CARREGAMENTO DO MODELO ---
if not os.path.exists('modelo_risco_saude.pkl'):
    with st.spinner('Treinando IA pela primeira vez...'):
        modelo = treinar_modelo()
else:
    modelo = joblib.load('modelo_risco_saude.pkl')

# --- 3. INTERFACE DO DASHBOARD ---
st.title("🏥 CardioPredict AI: Análise de Risco Cardíaco")
st.markdown("Preencha os dados abaixo para obter uma predição instantânea do risco de saúde.")

with st.form("diagnostico_form"):
    c1, c2, c3 = st.columns(3)
    
    with c1:
        st.subheader("📋 Perfil")
        idade = st.number_input("Idade", 18, 100, 35)
        sexo = st.selectbox("Sexo", ["Masculino", "Feminino"])
        hist_fam = st.selectbox("Histórico Familiar?", ["Não", "Sim"])
        trabalho = st.slider("Horas de Trabalho/Dia", 0, 16, 8)

    with c2:
        st.subheader("🍏 Hábitos")
        imc = st.number_input("IMC", 10.0, 50.0, 24.5)
        passos = st.number_input("Passos Diários", 0, 30000, 7000)
        sono = st.slider("Horas de Sono", 0, 12, 7)
        agua = st.slider("Água (Litros/Dia)", 0.0, 5.0, 2.0)

    with c3:
        st.subheader("🩺 Clínico")
        pressao = st.number_input("Pressão Sistólica", 80, 200, 120)
        colest = st.number_input("Colesterol", 100, 450, 180)
        fumante = st.selectbox("Fumante?", ["Não", "Sim"])
        alcool = st.selectbox("Álcool", ["Baixo", "Moderado", "Alto"])

    enviar = st.form_submit_button("🚀 Calcular Fator de Risco")

if enviar:
    # Processamento dos inputs para o formato do modelo
    f_num = 1 if fumante == "Sim" else 0
    a_num = {"Baixo": 0, "Moderado": 1, "Alto": 2}[alcool]
    s_num = 1 if sexo == "Masculino" else 0
    h_num = 1 if hist_fam == "Sim" else 0
    hip = 1 if pressao > 140 else 0
    imp_imc = imc * idade
    stress = 1 if (trabalho > 10 and sono < 6) else 0

    dados = pd.DataFrame([[
        idade, imc, passos, sono, agua, f_num, a_num, 
        pressao, hip, imp_imc, s_num, h_num, colest, stress
    ]], columns=[
        'Idade', 'IMC', 'Passos_Diarios', 'Horas_Sono', 'Agua_Litros', 
        'Fumante_Num', 'Alcool_Num', 'Pressao_Sistolica', 'Hipertensao', 
        'Impacto_IMC_Idade', 'Sexo_Num', 'Hist_Familiar_Num', 'Colesterol', 
        'Stress_Trabalho'
    ])

    # Predição
    resultado = modelo.predict(dados)[0]
    prob = modelo.predict_proba(dados).max()

    # Cores
    cor = {"Baixo": "green", "Moderado": "orange", "Alto": "red", "Muito Alto": "darkred"}[resultado]
    
    st.markdown(f"""
        <div style="background-color:{cor}; padding:25px; border-radius:10px; text-align:center;">
            <h2 style="color:white; margin:0;">RESULTADO: RISCO {resultado.upper()}</h2>
            <p style="color:white; font-size:1.1em;">Confiança do Modelo: {prob:.2%}</p>
        </div>
    """, unsafe_allow_html=True)

st.sidebar.markdown("---")
st.sidebar.write("✅ **Modelo Ativo:** RandomForest (300 árvores)")
st.sidebar.write("📊 **Acurácia Validada:** 81.80%")

2026-03-19 21:12:46.231 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 21:12:46.355 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 21:12:46.703 
  command:

    streamlit run c:\Projetos\pos ia\venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-19 21:12:46.704 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 21:12:46.704 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 21:12:46.705 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-19 21:12:46.705 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runni